# 📘 Notebook 07 · 链上量化 + MEV + 求职

> 最后一节。我们把前面的所有东西串起来：**链上量化 = 传统量化 + 智能合约 + 区块链特性**。

**学完你会：**

- 实现一个跨 DEX 套利策略（含代码 + 模拟）
- 理解 MEV：三明治攻击、抢跑、清算
- 知道 Flashbots、PBS、Builder 生态
- 看清链上量化 / Crypto Quant 的求职地图
- 知道 GSR、Wintermute、Jump Crypto 等顶级团队怎么进

**预计时间：8-10 小时**

## 1. 链上量化与传统量化的异同

### 1.1 一样的地方

- 数学和统计方法基本相同（多因子、协整、ML）
- 都看重数据基础设施、回测严谨度
- 都需要高水平工程能力

### 1.2 不一样的地方

| 维度 | 传统量化 | 链上 / Crypto 量化 |
|---|---|---|
| **市场结构** | 中心化撮合（NYSE、Nasdaq）| 公开 mempool + 链上结算 |
| **交易延迟** | 微秒（colocation）| 区块时间（ETH 12s、Solana 0.4s）|
| **手续费** | 万分之几 | 0.1% - 1%（gas）|
| **杠杆** | 受监管限制 | 永续 100x 唾手可得 |
| **数据** | 付费、Bloomberg、TAQ | 全部链上免费可读 |
| **可执行性** | 几乎瞬时 | 受 mempool / MEV 影响 |
| **机会** | 已经被高度挖掘 | 仍有大量低垂果实 |
| **风险** | 市场风险 | 市场 + 智能合约 + 协议失效 |
| **24/7** | 否 | 是（不停盘）|

### 1.3 三种主流链上量化方向

```
1. CEX-CEX / CEX-DEX 套利
   ├─ Binance vs Coinbase 现货价差
   ├─ 永续 vs 现货资金费率套利
   └─ CEX vs DEX 跨市场（最重要！）

2. 链上原生策略
   ├─ DEX 间套利（Uniswap vs Sushi）
   ├─ MEV：sandwich、liquidation
   ├─ 跨链桥套利
   └─ 新池子抢跑（snipe）

3. 高 alpha 链上做市
   ├─ Uniswap V3 LP 主动管理
   ├─ Curve / Balancer 流动性管理
   └─ 永续合约做市（dYdX、Hyperliquid）
```

## 2. 跨 DEX 套利：最入门的链上量化

### 2.1 原理

ETH/USDC 价格在不同 DEX 间不完全同步。

**例：**
- Uniswap V2 报价：1 ETH = 2000 USDC
- SushiSwap 报价：1 ETH = 1995 USDC

→ 在 Sushi 买入 ETH，在 Uniswap 卖出。利润 = 5 USDC / ETH（扣手续费 / Gas 前）

**真实世界中这种机会**：
- 平均存在时间 < 1 秒
- 利润率通常 0.05% - 0.2%
- 被 MEV bot 高度竞争

### 2.2 实现一个套利模拟器

In [ ]:
# 复用 Notebook 06 的 AMMPool（这里精简重写）
class AMM:
    def __init__(self, x, y, fee=0.003):
        self.x, self.y, self.fee = x, y, fee
    def price(self):
        return self.y / self.x if self.x > 0 else 0
    def amount_out(self, amount_in, side='x'):
        if side == 'x':
            xi, yo = self.x, self.y
        else:
            xi, yo = self.y, self.x
        a_in = amount_in * (1 - self.fee)
        return yo - xi * yo / (xi + a_in)
    def swap(self, amount_in, side='x'):
        out = self.amount_out(amount_in, side)
        if side == 'x': self.x += amount_in; self.y -= out
        else:           self.y += amount_in; self.x -= out
        return out


# 设两个 DEX，初始价格略不同
# pool_a: 100 ETH / 200,000 USDC → 价格 2000
# pool_b: 100 ETH / 199,500 USDC → 价格 1995（比 A 便宜 0.25%）
pool_a = AMM(x=100, y=200_000)     # x=ETH, y=USDC
pool_b = AMM(x=100, y=199_500)

print(f'初始：A 价格 {pool_a.price():.2f}, B 价格 {pool_b.price():.2f}')
print(f'差价: {(pool_a.price() - pool_b.price()) / pool_b.price() * 100:.3f}%')

In [ ]:
def find_arbitrage_size(pool_low, pool_high, gas_cost_usdc=20, max_size=10):
    """
    在 pool_low（便宜）买 ETH，在 pool_high（贵）卖 ETH。
    用二分搜最优规模：直到两边价格收敛或利润为负。
    返回 (最优买入 ETH 数量, 净利润)
    """
    best_size, best_profit = 0, 0
    # 用网格搜索（够用）
    for size in [0.01 * i for i in range(1, int(max_size * 100))]:
        # 模拟流程（不真实修改池子）
        # 在 pool_low 用 USDC 买 size ETH，需要多少 USDC？
        # 反向公式：要拿走 size ETH，需要付多少 USDC？
        # y_in_needed = x*y/(x - size) - y，再除以 (1-fee)
        x_low, y_low = pool_low.x, pool_low.y
        if size >= x_low * 0.99:
            break
        usdc_in = (x_low * y_low / (x_low - size) - y_low) / (1 - pool_low.fee)

        # 在 pool_high 卖 size ETH 得多少 USDC
        usdc_out = pool_high.amount_out(size, side='x')

        net_profit = usdc_out - usdc_in - gas_cost_usdc
        if net_profit > best_profit:
            best_size, best_profit = size, net_profit
    return best_size, best_profit

size, profit = find_arbitrage_size(pool_b, pool_a, gas_cost_usdc=20)
print(f'最优套利量: 买 {size:.4f} ETH')
print(f'净利润    : ${profit:.2f}')

if profit > 0:
    # 执行：用 USDC 在 pool_b 买 ETH，再用得到的 ETH 在 pool_a 卖
    # 反向算需要付多少 USDC（精确）
    usdc_in = (pool_b.x * pool_b.y / (pool_b.x - size) - pool_b.y) / (1 - pool_b.fee)
    # pool_b 是反向 swap：用 y 换 x
    eth_got = pool_b.swap(usdc_in, side='y')
    usdc_got = pool_a.swap(eth_got, side='x')
    print(f'\n实际执行: 投入 {usdc_in:.2f} USDC → 得到 {eth_got:.4f} ETH → 卖出得 {usdc_got:.2f} USDC')
    print(f'毛利: ${usdc_got - usdc_in:.2f}, 扣 gas 后: ${usdc_got - usdc_in - 20:.2f}')
    print(f'\n套利后池子状态：')
    print(f'  pool_a 价格: {pool_a.price():.2f}')
    print(f'  pool_b 价格: {pool_b.price():.2f}  （收敛了！）')

**关键观察**：

1. 套利后两个池子价格**收敛**——这就是为什么 DeFi 价格不会长期偏离
2. **gas 成本是硬约束**：以太坊主网 ~$20-50/笔，小套利根本不够付
3. **机会窗口极小**：你检测到机会 → 计算 → 提交交易 → 上链，全程需要 < 1 秒。慢一拍就被别人抢
4. 真实世界还有 **multi-hop 套利**（A→B→C→A），三角套利往往机会更多

### 2.3 真实做法的演进

```
v1：监控 mempool，看到大单就抢跑（被 MEV 玩烂了）
v2：用闪电贷做无本套利（资金效率 ∞）
v3：和 Flashbots 直接打交道（绕过公开 mempool）
v4：自建 Builder（不用 Flashbots 也能控制打包）
v5：跨链套利（ETH ↔ Arbitrum ↔ Base，桥的延迟是核心问题）
```

### 🎯 练习

三个练习：

1. 实现**三角套利**：USDC → ETH → DAI → USDC，找到任意 3 个池子，检测有没有循环套利机会
2. 加入 **slippage protection**：套利下单时设 minOutput，模拟"被抢跑导致价格变了"的情景
3. 用真实数据：从 Uniswap V2 / SushiSwap 主网读 ETH/USDC 池子状态（用 web3.py），计算理论套利空间

In [ ]:
# 在这里写你的答案


## 3. MEV：Maximal Extractable Value

### 3.1 是什么

MEV = **矿工 / 验证者通过控制交易包含、顺序、排除，能额外提取的价值**。

**为什么存在**：
- 区块链的"区块"内交易顺序可以由出块人随意调整
- 公开 mempool 让所有人看到待打包交易
- 智能合约可以原子地组合多个操作

**三大类 MEV：**

### 3.2 抢跑（Front-Running）

你看到一笔大单要在 Uniswap 买 100 ETH，你**抢先**也买 ETH，然后在它推高价格后卖出。

```
Mempool:
  [你的抢跑买单] @ gasPrice = 50 gwei      ← 先打包
  [受害者大买单] @ gasPrice = 30 gwei      ← 后打包，看到价格已被推高
  [你的卖出单]   @ gasPrice = 40 gwei      ← 再后打包
```

### 3.3 三明治攻击（Sandwich）

抢跑 + 反向"包"住受害者：

```
Block 第 N 个位置：
  ┌─ [你的买单：在受害者前] ───────┐
  │  [受害者：买 100 ETH]           ├─ 三明治
  └─ [你的卖单：在受害者后] ───────┘
```

利润 = 受害者推高的价差 - 两笔 gas - 自己交易的 AMM 手续费。

### 3.4 清算（Liquidation）

Aave、Compound 上抵押率不足的账户可被任何人清算。清算者得到 5-10% 折扣。

**MEV bot 们在比谁更快**：每个块都有几十个 bot 同时尝试，先到先得。

### 3.5 套利

如上一节所讲。真实的 DEX 价差套利大部分是 MEV bot 完成的。

### 3.6 MEV 数据

| MEV 类型 | 占比 | 备注 |
|---|---|---|
| 套利 | ~60% | 最大头 |
| 清算 | ~20% | 极端行情下飙升 |
| 三明治 | ~15% | 受害者主要是散户 |
| 其他 | ~5% | NFT mint 抢、新池子 snipe |

**数据来源**：[MEV-Explore](https://explore.flashbots.net)、[EigenPhi](https://eigenphi.io)、[Libmev](https://libmev.com)

历史上 MEV 总额 > $1B。

## 4. 模拟一次三明治攻击

为了让你**真正理解**，下面模拟一次完整的三明治攻击。

In [ ]:
# 重置一个池子
pool = AMM(x=100, y=200_000)
print(f'初始: ETH={pool.x}, USDC={pool.y}, price={pool.price():.2f}')

# ---------- 受害者要做的交易 ----------
# 受害者：拿 10,000 USDC 买 ETH。他设了 1% 的滑点容忍
victim_in = 10_000
expected_out = pool.amount_out(victim_in, side='y')
min_acceptable = expected_out * 0.99    # 滑点容忍 1%
print(f'\n[受害者] 计划用 {victim_in} USDC 买 ETH')
print(f'        预期得到 {expected_out:.6f} ETH')
print(f'        最低接受 {min_acceptable:.6f} ETH（滑点 1%）')

# ---------- 攻击者上场 ----------
# 攻击者看到这笔单在 mempool，决定夹他

# 第一步：抢在他前面买 ETH，把价格推高
attacker_in = 5_000
attacker_eth = pool.swap(attacker_in, side='y')
print(f'\n[攻击者-前手] 用 {attacker_in} USDC 抢先买，得 {attacker_eth:.6f} ETH')
print(f'              池子价格被推高到 {pool.price():.2f}')

# 第二步：受害者交易被打包（价格已变贵）
victim_actual = pool.swap(victim_in, side='y')
print(f'\n[受害者执行] 用 {victim_in} USDC 实际得到 {victim_actual:.6f} ETH')
print(f'             比预期少 {expected_out - victim_actual:.6f} ETH'
      f'（{(1 - victim_actual/expected_out)*100:.3f}%）')

# 如果损失 > 1% 滑点，交易会失败（被回滚）
if victim_actual < min_acceptable:
    print('             【交易回滚，攻击失败】')
else:
    print('             【交易成功，受害者损失被攻击者吃掉】')

# 第三步：攻击者卖出（价格已被两笔买单推高了）
attacker_usdc_back = pool.swap(attacker_eth, side='x')
profit = attacker_usdc_back - attacker_in
print(f'\n[攻击者-后手] 把 {attacker_eth:.6f} ETH 卖出，得 {attacker_usdc_back:.2f} USDC')
print(f'              毛利: ${profit:.2f}')
print(f'              扣 gas（假设 $30 双笔）: ${profit - 30:.2f}')

**关键点：**

1. **滑点容忍是受害者唯一的防御**——设太宽就给攻击者留空间，设太窄交易容易失败
2. **小额交易（< $1000）一般不会被夹**——攻击者赚不到 gas
3. **私有交易（Flashbots Protect）能防三明治**——直接发给 builder 不进公开 mempool
4. **不要在低流动性池子上做大单**——价格冲击大，三明治利润空间大

**搬到链上量化的角度**：
- 三明治不是黑客行为，是合法 MEV
- 但很多团队（如 Flashbots、CoW Swap）专门做反 MEV 工具
- 链上量化从业者**两边都做**：白天写抢跑 bot，晚上贡献开源反 MEV 工具

## 5. Flashbots 与 PBS 生态

### 5.1 Flashbots 的诞生

2020 年前，MEV 是 Wild West：
- 抢跑 bot 在公开 mempool 互相抬高 gas price，把网络挤崩
- 矿工自己也可能抢用户交易（trusted intermediary 失败）

[Flashbots](https://flashbots.net) 提出**MEV-Geth**（后来的 MEV-Boost）：
- searcher（找机会的人）把交易"打包"成 bundle
- 直接发给 builder / validator，绕过 mempool
- builder 看不到 bundle 内容（直到打包后），不能偷
- searcher 可以付费给 builder（priority fee）

效果：**MEV 提取从混乱的 mempool 战争变成了有序的拍卖**。

### 5.2 Proposer-Builder Separation (PBS)

以太坊合并（PoS）后，引入了 **PBS**：

```
Validator（提议者）            ←——— 卖区块空间
  │
  │ MEV-Boost 中继
  ▼
Builder（区块构建者）          ←——— 收 searcher 的 bundle
  │
  │ 接收 bundle + 排序
  ▼
Searcher（套利 / MEV 提取者）   ←——— 找机会
```

| 角色 | 功能 | 代表 |
|---|---|---|
| **Validator** | 出块，提议区块 | 所有 ETH staker |
| **Builder** | 构建区块（含 MEV）| beaverbuild、rsync-builder、Titan |
| **Relayer** | 中继 builder 和 validator | Flashbots Relay、Ultra Sound、Aestus |
| **Searcher** | 找 MEV 机会 | 各种 bot 团队 |

### 5.3 给链上量化从业者的影响

如果你想做 MEV 抢跑 / 套利：
- **不能再无脑发到 mempool 互卷 gas**
- 必须学会用 Flashbots / MEV-Share / mev-boost API
- 高级玩家**自建 Builder**，能看到所有 bundle，比别人快一步

如果你做"接受订单"型策略（如 LP 管理）：
- 你的交易**应该用 Flashbots Protect**，避免被夹
- 集成 CoW Swap 等 MEV-aware DEX

### 5.4 必读资源

- Flashbots Docs: https://docs.flashbots.net
- MEV-Explore: https://explore.flashbots.net
- Robert Miller 的 "Speculative Future of MEV"
- Yi Sun, Hasu 的 MEV 相关博客

## 6. 链上做市：Uniswap V3 LP 主动管理

链上做市（≈ Uniswap V3 LP 的主动管理）是另一种链上量化。

### 6.1 V3 LP 收益的拆解

$$\text{LP PnL} = \text{Fee income} - \text{Impermanent Loss} - \text{Gas cost}$$

- **Fee income**：在你的价格区间内的所有交易额 × 0.3%（或其他档位）× 你的流动性占比
- **IL**：价格变动导致的损失
- **Gas cost**：每次重新调整区间都要付 gas

### 6.2 主动管理策略

**策略 1：被动 wide range**
- 设很宽的区间（如 ±50%）
- 不需要频繁调整
- 收益 ≈ V2 LP

**策略 2：主动 narrow range + 重新平衡**
- 设很窄的区间（如 ±2%）
- 价格出区间立即重设
- 资金效率 50x，但 IL 更敏感

**策略 3：基于波动率定区间**
- 用 ETH 历史波动率算最优区间宽度
- 平衡 fee 和 IL

**策略 4：JIT (Just-In-Time) 流动性**
- 检测到大单要来，提前 mint LP 在窄区间
- 大单成交完立即 burn
- 0 IL 风险，但只在大单时赚

JIT 是高端玩家专属，需要 mempool 访问 + 极低延迟。

### 6.3 链上做市的真实团队

| 团队 | 类型 | 备注 |
|---|---|---|
| Arrakis Finance | LP 管理协议 | 用户存币，团队管理 |
| Gamma Strategies | LP 管理协议 | 类似 Arrakis |
| Wintermute | 做市商 | CEX + DEX 都做 |
| GSR | 做市商 | 老牌加密做市 |
| Cumberland | DRW 旗下 | 机构级做市 |
| Jump Crypto | Jump 旗下 | Solana 生态大户 |

## 7. 链上量化 / Crypto Quant 求职地图

### 7.1 三类雇主

**A. 传统量化 + 进入 crypto**

- Jump Crypto、Citadel Securities Crypto、Two Sigma、DRW (Cumberland)
- 文化、薪酬体系 ≈ 传统量化
- 起薪 80-200 万 RMB（应届硕士）
- 要求和传统量化几乎一样，额外加分项是熟悉 crypto

**B. 加密原生做市商 / 量化基金**

- Wintermute、GSR、Amber Group、QCP Capital、Auros
- 文化更 startup，灵活
- 起薪 50-150 万 RMB
- 看重 crypto 经验 + 工程能力

**C. 链上原生独立团队 / 工作室**

- 几个人组成的 MEV 团队、做市团队
- 工资不一定高，但**有 PnL 分成**
- 上限极高（个别团队个人年入 $10M+），下限可能为零
- 看重独立战斗能力，不要求名校学历

### 7.2 岗位类型

| 岗位 | 核心技能 | 起薪范围 |
|---|---|---|
| **链上 Quant Researcher** | Python + 数学 + 链上数据 | 80-150 万 |
| **MEV / Searcher** | Solidity + Rust/Go + 低延迟 | 100-300 万+ |
| **Smart Contract Engineer** | Solidity + 安全 | 60-150 万 |
| **DeFi Researcher** | Solidity + 经济学 + 数据 | 70-150 万 |
| **Crypto Data Engineer** | SQL + Python + 链上 | 40-100 万 |
| **Crypto Trading Engineer** | Python + Rust + 系统 | 60-150 万 |
| **Quant Strategy (Crypto)** | Python + 期权/永续 | 80-200 万 |

### 7.3 进入路径

**路径 1：传统量化 → Crypto**

最稳的路径：
1. 先在头部私募 / 投行做 2-3 年传统量化
2. 利用业余时间学 Solidity、玩链上数据
3. 投 Jump Crypto / Citadel / Wintermute
4. 起薪不会降，反而可能涨

**路径 2：DeFi 用户 → DeFi 从业**

适合 crypto 老玩家：
1. 深度使用 Uniswap、Aave、Curve 至少 1 年
2. 在 Dune 上写几个高质量 dashboard
3. 写 Medium 博客分析 DeFi 协议
4. 投 Wintermute、Amber、各种小团队
5. 起薪不高但成长快

**路径 3：CTF / 安全研究 → 智能合约**

如果你有黑客背景：
1. 玩 Ethernaut、Damn Vulnerable DeFi 全部通关
2. 在 Code4rena、Sherlock 上参与审计比赛
3. 拿过 $10k+ 的 bug bounty
4. 投 OpenZeppelin、Trail of Bits、ConsenSys Diligence
5. 安全审计师起薪 80-150 万

**路径 4：MEV / Searcher（最野的路径）**

适合天赋型 + 自驱型：
1. 自己 fork 公开 MEV bot 代码，跑起来
2. 在测试网模拟到主网，每天观察 mempool
3. 写一个 niche 套利 bot（如某个新 DEX、某个小币种）
4. 真的赚到钱，再放大
5. 全程不需要任何"职位"——你就是自己的老板

### 7.4 GSR、Wintermute、Jump Crypto 怎么进

**GSR**（Genesis Global Trading 衍生）：
- 校招通过官网 + LinkedIn
- 几轮技术面，重点考概率题和金融知识
- 注重团队文化匹配，沟通能力很重要

**Wintermute**：
- 主要内推 + 官网
- 看重 quant 背景 + crypto 经验
- 远程友好（Wintermute 是 100% 远程团队）

**Jump Crypto**：
- 母公司 Jump Trading 是顶级 HFT
- 校招体系成熟，但 crypto 部门更看实战
- 起薪可能是全行业最高

**common interview questions**：

1. "你最近一个月在追的 DeFi 协议？说说它的机制"
2. "Uniswap V3 和 V2 数学上的差异？"
3. "如何防止你的 MEV bot 被别人 reverse engineer？"
4. "Solana 和 Ethereum 的 MEV 生态差异？"
5. "你的某笔交易在 mempool 看到了，但被另一个 bot 抢了。怎么排查？"

如果你能在这些问题上给出**有质量的回答**（不需要完美），你就有 offer。

## 8. 链上量化的最终学习路线

如果你已经学完了 Notebook 01-06，再加上以下，你就是 **mid-level 链上量化研究员**：

```
基础                            来源
─────────────────────────────────────────
☐ 传统量化（Notebook 01-03）    本手册
☐ Solidity 实战                CryptoZombies、Ethernaut
☐ Uniswap V2 源码阅读           Uniswap GitHub
☐ Uniswap V3 数学               白皮书 + 解读博客

工具
─────────────────────────────────────────
☐ web3.py 流利使用              本手册 + 官方文档
☐ Foundry / Hardhat            官方教程
☐ Dune 写过 10+ 复杂 query     dune.com/learning
☐ The Graph 写过 subgraph      thegraph.com/docs
☐ Tenderly 调试链上交易         tenderly.co

实战
─────────────────────────────────────────
☐ 写过一个跨 DEX 套利 bot       本节
☐ 写过一个三明治模拟器          本节
☐ 跑过自己的 ERC20 + NFT       Notebook 05
☐ 在 Code4rena 提交过审计报告  code4rena.com
☐ 在 Dune 发布过 3+ dashboard  dune.com
☐ 在 Sepolia 部署过完整 DeFi 协议（mini Uniswap + 借贷）

加分
─────────────────────────────────────────
☐ 读 5+ 个主流协议的源码        Uniswap, Aave, Compound, Curve, Lido
☐ 复刻一个攻击事件分析报告      rekt.news 上的事件
☐ 自己 fork Reth / Erigon 跑节点  
☐ 写过 Rust 链上工具            alloy-rs、ethers-rs
☐ 在 Twitter 上有 follower      持续输出，1 年攒 1000+
```

## 9. 给你的最后一些建议

### 9.1 不要犯的错

❌ **追热点币种**：今天炒 meme，明天追 AI 板块——这不是研究，是赌博
❌ **太早做 MEV bot**：MEV 是顶级玩家的游戏，新手做亏的概率 95%
❌ **不学传统金融**：永续合约本质上是 90s 就有的金融衍生品，不学传统金融就理解不深
❌ **不写代码只看视频**：90% 的人停在"看了很多但什么也写不出来"
❌ **拒绝学英文**：Web3 90% 一手资料是英文，靠中文资料学是二手信息

### 9.2 要养成的习惯

✅ **每周读 3 篇 paper / 协议文档**：积少成多
✅ **每月写一篇博客 / 文章**：写作能力是稀缺的
✅ **在 Twitter / X 上关注 10 个核心人物**：Hasu、Tarun Chitra、Dan Robinson、Robert Miller
✅ **每周用 Dune 写一个 SQL**：链上数据感觉来自实战
✅ **每月跑一个新协议的测试网**：保持产品感觉

### 9.3 长期视角

> **Crypto 是一个 24/7 的全球市场，永远不缺机会**。 但**周期性极强**——熊市可能持续 2 年，期间所有人都怀疑自己。

熊市做什么：
- 学习、积累、不要花完上轮赚的钱
- 写代码、写文章、攒声誉
- **机会留给牛市没退场的人**

牛市做什么：
- 把熊市练好的东西兑现
- 控制贪婪，不上不该上的杠杆
- **资金安全永远第一**

### 9.4 资金安全（重要！！！）

**只要你做这一行，必须遵守的**：

1. **冷热钱包分离**：99% 资产放硬件钱包（Ledger / Trezor），只有少量在 MetaMask
2. **永远不在主网钱包测试合约**：用一个空钱包专门测试
3. **看清每个交易 approve 的额度**：默认 approve 是 ∞，是大量被盗根源
4. **不要把全部资产存在一个 CEX**：FTX 倒了，活生生 100 亿打水漂
5. **助记词写纸上，不要存电子版**：拍照、Google Doc、邮件都会被盗
6. **多签钱包是团队/大额必备**：Safe（前 Gnosis Safe）

## 10. 整套手册总结

走到这里，你应该已经：

```
量化交易                       Web3
═══════════════════════════════════════════════
✓ 多因子模型                   ✓ 区块链原理（手写过）
✓ IC 检验、分层回测            ✓ Solidity（写过 ERC20 + NFT）
✓ CTA、配对、ML 选股           ✓ web3.py（部署 + 交互）
✓ Walk-forward 严格回测        ✓ AMM 数学（手写过 Uniswap V2）
✓ 组合优化                     ✓ 借贷协议、闪电贷
✓ 国内私募 / 公募 / 投行         ✓ MEV、三明治、套利
✓ 简历、笔试、面试              ✓ Flashbots、PBS 生态
✓ 实习渠道                     ✓ 链上量化求职地图
```

**这套手册是骨架，肌肉要你自己长。** 接下来你应该做的：

1. **挑一个 niche 深挖**：要么传统量化方向，要么链上量化方向，**不要分散**
2. **做 1-2 个能写在简历最上面的项目**：质量 >> 数量
3. **建立你的"网络"**：找学长学姐、参加 meetup、Twitter 上和人对话
4. **保持 1-2 年的耐心**：第一份 offer 是最难的，之后会越来越顺

---

### 11. 进一步的资源

**量化：**
- 课程：Coursera 的"Investment Management with Python and Machine Learning Specialization"
- 论文：SSRN 关注 Marcos López de Prado、Bryan Kelly、Stefan Nagel
- 中文社区：知乎"量化投资"、聚宽、米筐

**Web3：**
- Andreas Antonopoulos 的《Mastering Ethereum》（GitHub 免费）
- Cyfrin Updraft（YouTube 频道）：Patrick Collins 的 Solidity 教程
- Paradigm Research 博客
- 0xCygaar、Hasu、Tarun Chitra 的 Twitter

**通用：**
- Patrick McKenzie 关于职业的博客（patio11.com）
- "The Art of Doing Science and Engineering" by Hamming
- "Skin in the Game" by Taleb（金融业必读）

---

**最后一句话：**

> 量化交易和 Web3 本质上都是一件事——**用代码和数学，在嘈杂的市场里找到一个微小但可重复的优势**。**找到它不容易，守住它更难。** 你需要的是：扎实的技术、持续的好奇、对风险的敬畏、不被牛市冲昏头的耐心。

> 这本手册到这就结束了。**祝你 PnL 长虹。**

---
*《量化交易 × Web3 学习手册》全 8 个 Notebook 完结*